# 06. FOL Inference: Unification, Forward & Backward Chaining

Notebook 05 membangun KB Colonel West sebagai FOL sentence, tapi belum
membuktikan apa pun darinya. Notebook ini melengkapi sisi inferensinya:
**unification** sebagai mekanisme pencocokan pattern di balik semua
algoritma FOL, lalu **forward** dan **backward chaining** versi FOL --
generalisasi langsung dari Horn-clause forward/backward chaining di
Notebook 04, sekarang untuk rule yang predicate-nya punya variable.

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. menjelaskan universal instantiation secara singkat sebagai jembatan ke
   unifikasi;
2. menjalankan algoritma UNIFY untuk mencari substitution yang menyamakan
   dua sentence, termasuk kenapa occurs check diperlukan;
3. menelusuri FOL forward chaining (Generalized Modus Ponens) pada KB
   Colonel West sampai membuktikan `Criminal(West)`;
4. menelusuri FOL backward chaining pada KB yang sama, dan
   membandingkannya dengan forward chaining; dan
5. membangun sendiri sebuah reasoning engine kecil (forward dan backward)
   di atas KB baru sebagai latihan akhir.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : C:\Users\cathl\Kuliah\KCV\KK\Modul-Praktikum-KK-RKA-25\logics\praktikum\environment
Python      : 3.14.2
Check       : tt_entails(P & Q, Q) = True


---
# 6.1 Universal Instantiation (Singkat)

## Penjelasan

Bagaimana persisnya $\forall x\ \alpha$ dipakai untuk inferensi? Lewat
**Universal Instantiation**: dari $\forall v\ \alpha$, boleh disimpulkan
$\text{Subst}(\{v/g\}, \alpha)$ untuk **sembarang** ground term $g$ (term
yang tidak mengandung variable). Contoh dari slide:

$$\forall x\ \text{King}(x) \land \text{Greedy}(x) \Rightarrow
\text{Evil}(x)$$

meng-entail *setiap* instantiation-nya sekaligus: `King(John) &
Greedy(John) ==> Evil(John)`, `King(Richard) & Greedy(Richard) ==>
Evil(Richard)`, dan seterusnya untuk tiap object yang ada.

Algoritma FOL di sisa notebook ini (`fol_fc_ask`, `fol_bc_ask`)
menerapkan instantiation ini otomatis di balik layar, dikombinasikan
dengan unifikasi (6.2) supaya tidak perlu mencoba **setiap** object satu
per satu secara membabi buta. Existential instantiation (untuk
$\exists$) juga ada, tapi karena `logic.py` tidak punya syntax $\exists$
(Notebook 05, 5.3) dan cakupan notebook ini cuma forward/backward
chaining atas definite clause, bagian itu dilewati.

## Contoh penerapan

`subst(s, x)` adalah fungsi yang benar-benar melakukan penggantian
variable itu -- persis mekanisme di balik Universal Instantiation.

In [2]:
psource(subst)

rule = expr('(King(x) & Greedy(x)) ==> Evil(x)')
grounded = subst({expr('x'): expr('John')}, rule)

print("Sebelum :", rule)
print("Sesudah :", grounded)

Sebelum : ((King(x) & Greedy(x)) ==> Evil(x))
Sesudah : ((King(John) & Greedy(John)) ==> Evil(John))


---
# 6.2 Unification

## Penjelasan

**Unification** adalah kebalikan dari instantiation: bukan memasukkan
object tertentu ke sebuah variable, tapi **mencari** substitution yang
membuat dua sentence menjadi identik. $\text{UNIFY}(p, q) = \theta$
berarti $\text{SUBST}(\theta, p) = \text{SUBST}(\theta, q)$, atau `None`
kalau tidak ada $\theta$ seperti itu.

Ada satu penjagaan penting, **occurs check**: sebuah variable tidak
boleh disubstitusi dengan term yang memuat variable itu sendiri
(misalnya `x` dengan `F(x)`) -- kalau dibolehkan, hasilnya struktur tak
berhingga.

## Contoh penerapan

Empat contoh dari slide, semuanya query terhadap KB $\forall x\
\text{Knows}(x, \text{Obama})$:

In [3]:
psource(unify, occur_check)

In [4]:
print(unify(expr('Knows(x, Obama)'), expr('Knows(Steve, Obama)')))
print(unify(expr('Knows(x, Obama)'), expr('Knows(Bill, y)')))
print(unify(expr('Knows(x, Obama)'), expr('Knows(Mother(y), y)')))
print(unify(expr('Knows(x, Obama)'), expr('Knows(Elizabeth, x)')))

{x: Steve}
{x: Bill, y: Obama}
{x: Mother(Obama), y: Obama}
None


Tiga contoh pertama berhasil: `{x: Steve}`, `{x: Bill, y: Obama}`, dan
`{y: Obama, x: Mother(Obama)}` -- ketiganya benar-benar membuat kedua
sisi identik kalau disubstitusi. Contoh terakhir gagal (`None`): argumen
kedua di kiri (`Obama`) harus disamakan dengan argumen kedua di kanan,
yang sudah terikat ke `Elizabeth` dari argumen pertama -- `Obama` dan
`Elizabeth` dua constant berbeda, tidak bisa disatukan.

Occurs check dicontohkan terpisah supaya jelas bukan itu sebab kegagalan
di atas:

In [5]:
# x tidak boleh disubstitusi dengan term yang memuat x sendiri.
print(unify(expr('x'), expr('F(x)')))

None


---
# 6.3 Generalized Modus Ponens & Forward Chaining

## Penjelasan

**Generalized Modus Ponens** (GMP) menggeneralisasi Modus Ponens
(Notebook 04, 4.1) supaya bekerja dengan predicate ber-variable, lewat
unifikasi:

$$\frac{p_1', p_2', \dots, p_n', \quad (p_1 \land \dots \land p_n
\Rightarrow q)\ \ \text{dengan}\ \ \text{UNIFY}(p_i', p_i) = \theta\
\text{untuk semua}\ i}{\text{SUBST}(\theta, q)}$$

Sama seperti Modus Ponens biasa (satu premise cocok, simpulkan
konklusi), cuma sekarang "cocok" berarti "bisa di-unify dengan
substitution $\theta$ yang **sama** untuk seluruh premise," bukan
sekadar sama persis.

**FOL-FC-ASK** (Figure 9.3, `fol_fc_ask` di `logic.py`) menerapkan GMP
berulang: cari semua kombinasi rule + substitution yang seluruh
premise-nya terpenuhi lewat fact yang sudah ada di KB, tambahkan
konklusinya, ulangi sampai tidak ada fact baru atau query terjawab.
Fungsinya berupa *generator*, jadi perlu dibungkus `list()` untuk
melihat semua jawabannya.

## Contoh penerapan

Buktikan `Criminal(West)` dari `crime_kb` (Notebook 05, 5.4). Karena
`fol_fc_ask` menambahkan tiap fact baru yang ditemukannya langsung ke KB
lewat `kb.tell(...)`, sel di bawah memakai salinan segar `crime_kb`
supaya aman dijalankan berulang kali tanpa fact-nya menumpuk.

In [6]:
def fresh_crime_kb():
    """A fresh copy of crime_kb. fol_fc_ask below calls kb.tell() for
    every new fact it derives, so reusing the same FolKB object across
    re-runs of this notebook would keep piling facts on top of the
    previous run."""
    return FolKB(crime_kb.clauses)

In [7]:
psource(fol_fc_ask)

In [8]:
kb = fresh_crime_kb()
before = len(kb.clauses)

answers = list(fol_fc_ask(kb, expr('Criminal(x)')))

print("Jawaban          :", answers)
print("Jumlah clause KB :", before, "->", len(kb.clauses))

Jawaban          : [{x: West}]
Jumlah clause KB : 8 -> 12


Jawabannya `[{x: West}]`: `Criminal(West)` terbukti, persis target yang
dinyatakan di 5.4. Jumlah clause KB juga bertambah -- itu semua fact
antara yang ikut diturunkan sepanjang jalan (`Weapon(M1)`,
`Sells(West, M1, Nono)`, `Hostile(Nono)`, dan `Criminal(West)` sendiri),
sama seperti trace di slide: dari `Missile(M1)` didapat `Weapon(M1)`,
dari `Missile(M1) & Owns(Nono, M1)` didapat `Sells(West, M1, Nono)`,
dari `Enemy(Nono, America)` didapat `Hostile(Nono)`, dan begitu keempatnya
lengkap untuk `x = West`, rule kejahatan menyala.

---
# 6.4 Backward Chaining

## Penjelasan

**FOL-BC-ASK** (Figure 9.6, `fol_bc_ask`/`fol_bc_or`/`fol_bc_and`)
adalah generalisasi backward chaining Notebook 04 (4.4) ke FOL:
goal-driven, cari rule yang konklusinya cocok (lewat unifikasi, bukan
sekadar `==`), lalu buktikan tiap premise-nya secara rekursif sebagai
sub-goal. Bedanya dengan versi propositional, tiap kali sebuah rule
dipakai, variable-nya di-*standardize* dulu (`standardize_variables`)
supaya tidak bentrok kalau rule yang sama dipakai dua kali di jalur
pembuktian yang berbeda.

Menelusuri KB Colonel West secara mundur dari `Criminal(West)` (slide):
buktikan `American(West)` (langsung fact), `Weapon(y)`, `Sells(West, y,
Nono)`, dan `Hostile(Nono)`. `Weapon(y)` mundur lagi ke `Missile(y)`;
`Sells(West, y, Nono)` mundur ke `Missile(y) & Owns(Nono, y)`;
unifikasi menyatukan `y = M1` di kedua cabang; `Hostile(Nono)` mundur ke
`Enemy(Nono, America)` (fact). Semua premise terbukti, kesimpulan yang
sama dengan forward chaining tercapai lewat jalur yang berbeda.

## Contoh penerapan

In [9]:
psource(fol_bc_ask)

In [10]:
# Query ground (sudah lengkap, tidak ada variable): jawabannya cuma
# konfirmasi terbukti/tidak.
print(list(fol_bc_ask(fresh_crime_kb(), expr('Criminal(West)'))))

# Query dengan variable: fol_bc_ask juga bisa mencari siapa saja x-nya,
# sama seperti fol_fc_ask di 6.3.
print(list(fol_bc_ask(fresh_crime_kb(), expr('Criminal(x)'))))

[{v_0: West, v_13: M1, v_1: M1, v_20: M1, v_2: Nono, v_32: Nono}]
[{v_54: West, x: West, v_67: M1, v_55: M1, v_74: M1, v_56: Nono, v_86: Nono}]


Hasilnya lebih berantakan dibanding `fol_fc_ask` di 6.3: bukan cuma
`{x: West}`, tapi satu dict besar berisi banyak nama sementara seperti
`v_54`, `v_67`, dan seterusnya, berdampingan dengan `x`. Itu bukan bug --
`v_N` adalah variable baru yang dibuat `standardize_variables` tiap kali
sebuah rule dipakai di sepanjang jalur pembuktian (Penjelasan di atas),
dan `fol_bc_ask` tidak membersihkannya dari hasil akhir. Yang penting
dicari cuma satu: `West` tetap muncul di situ, dipetakan dari `x` --
kesimpulan yang sama dengan `fol_fc_ask`, cuma dibungkus lebih
berantakan.

Query pertama (`Criminal(West)`, sudah ground) hasilnya serupa: sebuah
dict penuh `v_N`, tanpa `x` sama sekali (karena query-nya sendiri memang
tidak punya variable untuk dipetakan). Yang penting bukan isi dict-nya,
tapi bahwa `fol_bc_ask` menghasilkan **sedikitnya satu** substitution
sama sekali, bukan list kosong -- bandingkan Soal 3 di Latihan Soal, yang
query gagalnya justru menghasilkan list kosong.

In [11]:
from notebook import Canvas_fol_bc_ask

Canvas_fol_bc_ask('canvas_bc_ask', fresh_crime_kb(), expr('Criminal(x)'))

Kanvas di atas menggambar AND-OR proof tree hasil backward chaining:
tiap kotak satu goal/sub-goal (teksnya mungkin terpotong karena
kotaknya kecil). Klik kotak mana pun untuk menampilkan teks lengkapnya
di bar bawah -- cara lain melihat jalur pembuktian yang sama dengan
yang dijelaskan di atas, tanpa harus membaca ulang trace secara
manual.

---
# 6.5 Forward vs Backward (FOL)

## Penjelasan

Perbandingannya sama persis dengan Notebook 04 (4.4), sekarang di level
FOL:

| | Forward chaining | Backward chaining |
|---|---|---|
| Titik mulai | Fact | Query (goal) |
| Arah | Fact $\to$ goal | Goal $\to$ fact |
| Menambah clause ke KB? | Ya (`kb.tell` tiap fact baru) | Tidak |
| Cocok dipakai kalau | KB kecil-menengah, banyak query berbeda | KB besar, query spesifik dan sedikit |

Ada metode ketiga yang sepenuhnya general, **resolution**, yang tidak
terikat ke bentuk definite clause sama sekali (beda dengan
forward/backward chaining di sini) -- disinggung di Notebook 04 (4.2),
tapi di luar cakupan modul ini.

## Contoh penerapan

Bandingkan langsung lewat jumlah clause yang bertambah di KB,
mengonfirmasi baris ketiga tabel di atas.

In [12]:
kb_fc = fresh_crime_kb()
list(fol_fc_ask(kb_fc, expr('Criminal(x)')))
print("Forward chaining : KB tumbuh dari", len(fresh_crime_kb().clauses),
      "jadi", len(kb_fc.clauses), "clause.")

kb_bc = fresh_crime_kb()
list(fol_bc_ask(kb_bc, expr('Criminal(x)')))
print("Backward chaining: KB tetap", len(kb_bc.clauses), "clause.")

Forward chaining : KB tumbuh dari 8 jadi 12 clause.
Backward chaining: KB tetap 8 clause.


---
# Latihan Soal

## Soal 1

Forward chaining versi propositional (Notebook 04, 4.3) langsung
memproses fact dari `agenda`. Forward chaining versi FOL (6.3) butuh
mekanisme tambahan: mencoba berbagai kombinasi constant untuk tiap
variable rule (`enum_subst` di dalam `fol_fc_ask`, kelihatan lewat
`psource`). Jelaskan dengan kata-kata sendiri kenapa versi FOL butuh
langkah ekstra ini padahal versi propositional tidak.

<details>
<summary>Klik untuk melihat hint</summary>

Bandingkan bentuk fact di 04 (`A`, `B` -- symbol polos) dengan fact di
sini (`American(West)` -- ada constant di dalam predicate). Sekarang
lihat rule `American(x) & Weapon(y) & ... ==> Criminal(x)`: apa rule ini
bisa langsung dibandingkan "sama persis" dengan sebuah fact seperti
`American(West)`? Apa yang perlu dicari dulu supaya `x` di rule dan
`West` di fact bisa "dianggap sama" (6.2)? Itulah kombinasi constant
yang dicoba `enum_subst`.

</details>

## Soal 2

Bangun knowledge base baru (domain berbeda dari Colonel West) buat
sistem kelayakan praktikum kampus:

```text
Student(Ani), Takes(Ani, AI), Passed(Ani, Pemrograman)
Student(Budi), Takes(Budi, AI)
(Student(x) & Takes(x, AI) & Passed(x, Pemrograman)) ==> Eligible(x, AILab)
```

Budi sengaja belum dikasih fact `Passed(Budi, Pemrograman)`. Bangun
sebagai `FolKB`, lalu jalankan `fol_fc_ask` untuk query `Eligible(x,
AILab)`. Pastikan cuma `Ani` yang muncul di jawaban.

<details>
<summary>Klik untuk melihat hint</summary>

Ikuti pola membangun KB di 5.4: `FolKB` menerima list `expr(...)`, fact
dan rule campur jadi satu list. Tulis 5 fact (3 untuk Ani, 2 untuk Budi)
plus 1 rule kelayakan, persis seperti tertulis di soal -- lalu jalankan
`fol_fc_ask` sama seperti 6.3. Kalau jawabannya masih memuat `Budi`, cek
lagi: fact mana yang sengaja belum ditambahkan?

</details>

## Soal 3

Masih dengan KB dari Soal 2, jalankan `fol_bc_ask` dua kali secara
terpisah: sekali untuk query `Eligible(Ani, AILab)`, sekali untuk
`Eligible(Budi, AILab)`. Jelaskan lewat hasilnya (bukan cuma
`True`/`False`, tapi apa arti list kosong vs list berisi) kenapa satu
berhasil dan satu tidak.

<details>
<summary>Klik untuk melihat hint</summary>

Catatan teknis dulu, bukan jawabannya: kalau hasil `fol_bc_ask` kelihatan
"berantakan" (banyak `v_N` bercampur di satu substitution, kadang lebih
dari satu entri di list), itu wajar -- lihat catatan `standardize_variables`
di 6.4. Yang perlu dicek cuma satu hal: **list-nya kosong atau tidak**,
bukan menghitung persis isinya.

Kalau salah satu query hasilnya list kosong: itu generator yang tidak
pernah menemukan satu jalur pembuktian pun. Telusuri lagi rule
kelayakan-nya premise per premise untuk mahasiswa itu (sama seperti
Soal 2) -- premise mana yang macet?

</details>

## Soal 4

Untuk KB kelayakan praktikum di Soal 2-3: kalau tujuannya adalah
mencetak **daftar semua mahasiswa yang eligible**, mana yang lebih
efisien dipanggil, forward chaining atau backward chaining? Bagaimana
kalau tujuannya cuma mengecek **satu mahasiswa tertentu** saja?
Jelaskan alasannya, kaitkan dengan tabel perbandingan di 6.5, lalu --
sebagai penutup notebook ini -- tulis satu paragraf yang menjelaskan ke
pembaca awam (natural language, bukan notasi FOL) kenapa `Ani` eligible
ikut praktikum AI, berdasarkan fact dan rule yang ada di KB-nya.

<details>
<summary>Klik untuk melihat hint</summary>

Bandingkan: `fol_fc_ask(campus_kb, expr('Eligible(x, AILab)'))` dengan
`x` sebagai variable dijalankan berapa kali untuk dapat **semua** yang
eligible? Sekarang bandingkan dengan `fol_bc_ask` -- kalau mau tahu
status Budi doang, apa perlu tahu status Ani dulu? Kaitkan jawabanmu ke
baris "cocok dipakai kalau" di tabel 6.5, dan ke kesimpulan Notebook 04
Soal 4 (pola perbandingannya sama, cuma levelnya beda).

Untuk paragrafnya: coba susun murni dari fact yang ada di KB (`Student`,
`Takes`, `Passed`) tanpa nyebut kata "FOL", "predicate", atau notasi
apa pun -- seolah menjelaskan ke teman yang belum pernah belajar logika
sama sekali.

</details>